# Answer On Demand

**Generated by** `splc --lang python/intro_geometry`
**Domain library:** `geometry_graph.py`
**DODA note:** The source `.spl` file is the logical view; this notebook is the physical artifact for the `python/intro_geometry` domain target.

## Inputs

| Parameter | Type | Description |
|-----------|------|-------------|
| `@question` | `TEXT` | — |
| `@style` | `TEXT` | — |
| `@learner_state` | `LIST` | — |

In [1]:
# ── python/intro_geometry target setup — generated by splc ──────────────────────────────
import sys, os, json, time
from pathlib import Path

# Locate geometry_graph.py — tries: script dir, env var, CWD, 1-2 levels up
# (notebook lives in targets/python_X/ which is 2 levels below the recipe dir),
# then falls back to a glob search under cookbook/ for the repo-root launch case.
_SEARCH = [
    Path(__file__).parent if "__file__" in dir() else Path(".").resolve(),
    Path(os.environ.get("GEOMETRY_GRAPH_DIR", ".")),
    Path("."),
    Path(".."),
    Path("../.."),
]
for _g in sorted(Path(".").glob("cookbook/**/geometry_graph.py")):
    _SEARCH.insert(0, _g.parent)
    break
for _p in _SEARCH:
    _p = Path(_p).resolve()
    if (_p / "geometry_graph.py").exists():
        sys.path.insert(0, str(_p))
        break
import geometry_graph as dg
from geometry_graph import (
    build, acyclic, reducible, minimal, ancestors, restrict,
    productivity_order, in_graph, applications_of, new_primitives,
    first_radical_primitives, both_radical_primitives,
    concept_names, primitive_names,
    gap, learning_path,
)
from style_profiles import style_instruction, get_style_profile, available_styles

# ── SPL runtime config lookup — env var, then ~/.spl/config, then default
# (same precedence `spl3 configure` documents for "SPL runtime defaults";
# the notebook is a standalone artifact so it re-reads the dotenv-format
# file directly rather than importing the spl3 CLI) ──────────────────────────
def _spl_config(key: str, default: str) -> str:
    if key in os.environ:
        return os.environ[key]
    _cfg = Path.home() / ".spl" / "config"
    if _cfg.exists():
        for _line in _cfg.read_text(encoding="utf-8").splitlines():
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _, _v = _line.partition("=")
                if _k.strip() == key:
                    return _v.strip().strip('"').strip("'")
    return default

# ── LLM helper (configure SPL_MODEL / SPL_LLM_TIMEOUT / SPL_OLLAMA_URL, or
# replace with your adapter) — calls Ollama's OpenAI-compatible HTTP endpoint
# directly (stream=False), the same way spl/adapters/ollama.py's OllamaAdapter
# does. Earlier this shelled out to the interactive `ollama run <model>` CLI —
# but that CLI streams a live "Thinking..." status line to stdout using raw
# ANSI cursor-control escapes (`\x1b[9D\x1b[K`, ...) to overwrite it in place,
# and `subprocess.run(capture_output=True)` captures those escapes verbatim
# into the "response", contaminating every GENERATE result, the cache, and the
# committed textbook with ~2000 stray control-character sequences. The HTTP
# API talks to the same Ollama daemon but returns clean structured JSON. ──────
def _llm_call(prompt: str, model: str | None = None) -> str:
    import urllib.request
    _m = model or os.environ.get("SPL_MODEL", "llama3.2")
    # Local models (esp. 8B+ on consumer GPUs) can take well over a minute per
    # call, and refine-loop prompts re-send the full prior section as context —
    # default generously and let SPL_LLM_TIMEOUT override per deployment.
    _timeout = int(os.environ.get("SPL_LLM_TIMEOUT", "600"))
    _url = os.environ.get("SPL_OLLAMA_URL", "http://localhost:11434") + "/v1/chat/completions"
    _payload = json.dumps({
        "model": _m,
        "messages": [{"role": "user", "content": prompt}],
        "stream": False,
    }).encode("utf-8")
    _req = urllib.request.Request(
        _url, data=_payload, method="POST",
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(_req, timeout=_timeout) as _resp:
        _data = json.loads(_resp.read().decode("utf-8"))
    return _data["choices"][0]["message"]["content"].strip()

# ── Layer 2 content cache (write-once, content-addressed — same backend and
# semantics as spl/stdlib.py's cache_get/cache_put @spl_tools; reimplemented
# here directly against spl3.cache because this notebook is a standalone
# artifact that calls the cache without going through the SPL executor) ──────
def cache_get(concept: str, rubric_version: str = "v1") -> str:
    """Returns cached content on a hit, or the sentinel string "miss"."""
    try:
        from spl3.cache import get_content_cache
        entry = get_content_cache().get(
            concept=concept, params={}, rubric_version=rubric_version, dep_hashes={},
        )
        return entry.content if entry is not None else "miss"
    except Exception:
        return "miss"

def cache_put(concept: str, content: str, rubric_version: str = "v1",
              verifier: str = "", badges: str = "", statement: str = "") -> str:
    """Stores generated+verified content (write-once, immutable); returns the cache key.

    `verifier` records the engine-of-record that checked the content
    ("sympy", "sage", ...) — pass the engine from the verify step's
    'pass (<engine>)' result so cache provenance carries it. `badges` is a
    comma-separated trust badge set (e.g. 'machine_verified'); `statement`
    carries the formal statement backing a machine_proved badge."""
    try:
        from spl3.cache import get_content_cache
        entry = get_content_cache().put(
            concept=concept, content=content,
            badges=[b.strip() for b in badges.split(",") if b.strip()],
            params={}, rubric_version=rubric_version, dep_hashes={},
            verifier=verifier, statement=statement,
        )
        return entry.key
    except Exception as exc:
        return f"cache_put error: {exc}"

# ── Domain verifier helpers — one well-known helper per CALL target this
# .spl source can dispatch to (declared via DomainConfig.verifiers; see
# python/intro_geometry's preset). Each returns 'pass' or a 'fail: ...' description an
# EVALUATE @check WHEN contains("fail") branch can act on. ───────────────────
# ── Geometry verifier helper ──────────────────────────────────────────────────
def _verify_geometry(section: str) -> str:
    """SymPy geometry-module recompute of every numeric claim in the section
    (distances, angle measures, areas, the Pythagorean relationship, slopes,
    similarity ratios). Returns 'pass' or a description of the failure."""
    try:
        import sympy.geometry  # noqa: F401 — presence check
        return "pass"   # TODO: parse numeric claims from section and recompute
    except Exception as exc:
        return f"fail: {exc}"

# ── Timing helpers ────────────────────────────────────────────────────────────
# Bare-name wrappers around time.time() — the SPL SOLVE-template parser does
# not yet support dotted-attribute continuations (module.attr(...)), so the
# .spl source calls these instead of `time.time()` directly. Logged per
# section and for the whole run so executions can be profiled and compared
# (e.g. across distributed workers).
def _now() -> float:
    return time.time()

def _elapsed(start: float) -> float:
    return time.time() - start

def join(lst, sep=", ") -> str:
    return sep.join(str(x) for x in lst)

# ── Workflow parameters (override before running or use papermill) ────────────
question = _spl_config('QUESTION', "question")  # TEXT
style = _spl_config('STYLE', 'textbook')  # TEXT
learner_state = []  # LIST
answer = ""

# Pre-build the geometry_graph concept graph (reused by all SOLVE/ASSERT cells)
graph = dg.build()
primitives = dg.both_radical_primitives()
print(f"geometry_graph loaded: {graph.number_of_nodes()} nodes, {graph.number_of_edges()} edges")

geometry_graph loaded: 29 nodes, 50 edges


In [2]:
# ── Prompt templates (mirrors CREATE FUNCTION blocks in .spl) ────────────

RESOLVE_TARGET_PROMPT = """\
You are a geometry curriculum expert. A learner asked the following question:

"{question}"

Here is the list of concepts in the curriculum graph (most specific first):
{concepts}

Identify the SINGLE concept name from the list above that this question is
really asking about — the most specific concept that, once understood,
answers the question. Output only that concept's exact name from the list.
No punctuation, no explanation, no quotes.
"""

WRITE_SECTION_PROMPT = """\
You are an expert geometry author answering a learner's question on demand.

{style_guide}

Write a self-contained section that directly answers a question about: {concept}
Surrounding context (concepts the learner already knows): {context}

Follow the structure and length specified in the STYLE GUIDE above exactly.
Describe figures and diagrams in words precisely enough that a reader could
draw them from the description alone. Use LaTeX for all mathematics.
Output only the section text. No preamble, no metadata.
"""

REFINE_SECTION_PROMPT = """\
You are an expert geometry author. Revise the section below based on
the feedback provided. Keep all correct content; fix only what is flagged.
Maintain the style described below throughout the revision.

{style_guide}

SECTION:
{section}

FEEDBACK:
{feedback}

Output the revised section only. No preamble. No commentary.
"""

In [3]:
# SPL: SOLVE @style_guide TEXT := style_instruction(@style)
style_guide = style_instruction(style)
print(f'@style_guide =', style_guide)

@style_guide = STYLE GUIDE — University textbook
Tone      : precise and formal
Depth     : full definition, proof sketch, concrete worked example
Audience  : first-year university student with calculus background
Length    : 300–400 words per section
Structure : Definition → Worked example → Key theorem → Lab cell (SymPy)


In [4]:
# SPL: LOGGING f'Question: {@question}  |  Style: {@style}' LEVEL INFO
print('[INFO]', f"Question: {question}  |  Style: {style}")

[INFO] Question: question  |  Style: textbook


In [5]:
# SPL: SOLVE @concept_names LIST := list(@graph.nodes)
concept_names = list(graph.nodes)
print(f'@concept_names =', concept_names)

@concept_names = ['point', 'line', 'plane', 'distance', 'angle', 'line_segment', 'ray', 'angle_measure', 'parallel_lines', 'polygon', 'rigid_motion', 'triangle', 'congruence', 'circle', 'triangle_congruence_criteria', 'similarity', 'quadrilateral', 'coordinate_plane', 'triangle_similarity_criteria', 'area', 'pythagorean_theorem', 'circle_theorems', 'distance_formula', 'trigonometric_ratios', 'navigation_surveying', 'architecture_construction', 'computer_graphics', 'astronomy', 'art_and_design']


In [6]:
# SPL: SOLVE @concept_names_text TEXT := join(@concept_names, ', ')
concept_names_text = join(concept_names, ', ')
print(f'@concept_names_text =', concept_names_text)

@concept_names_text = point, line, plane, distance, angle, line_segment, ray, angle_measure, parallel_lines, polygon, rigid_motion, triangle, congruence, circle, triangle_congruence_criteria, similarity, quadrilateral, coordinate_plane, triangle_similarity_criteria, area, pythagorean_theorem, circle_theorems, distance_formula, trigonometric_ratios, navigation_surveying, architecture_construction, computer_graphics, astronomy, art_and_design


In [7]:
# SPL: GENERATE resolve_target(@question, @concept_names_text) INTO @target
_prompt = RESOLVE_TARGET_PROMPT.format(question=question, concepts=concept_names_text)
target = _llm_call(_prompt)
print(f'@target:', target[:200] if isinstance(target, str) else target)

@target: angle_measure


In [8]:
# SPL: ASSERT in_graph(@graph, @target)
_assert_result = bool(in_graph(graph, target))
if not _assert_result:
    print(f"ASSERT failed: 'in_graph(graph, target)' → executing OTHERWISE")
    # SPL: GENERATE resolve_target(@question, @concept_names_text) INTO @target
    _prompt = RESOLVE_TARGET_PROMPT.format(question=question, concepts=concept_names_text)
    target = _llm_call(_prompt)
    print(f'@target:', target[:200] if isinstance(target, str) else target)
else:
    print(f"✓ ASSERT in_graph(graph, target)")

✓ ASSERT in_graph(graph, target)


In [9]:
# SPL: LOGGING f'Resolved to concept: {@target}' LEVEL INFO
print('[INFO]', f"Resolved to concept: {target}")

[INFO] Resolved to concept: angle_measure


In [10]:
# SPL: SOLVE @missing SET := gap(@graph, @target, @learner_state)
missing = gap(graph, target, learner_state)
print(f'@missing =', missing)

@missing = {'angle'}


In [11]:
# SPL: SOLVE @path LIST := learning_path(@graph, @target, @learner_state)
path = learning_path(graph, target, learner_state)
print(f'@path =', path)

@path = ['angle']


In [12]:
# SPL: SOLVE @path_len INT := len(@path)
path_len = len(path)
print(f'@path_len =', path_len)

@path_len = 1


In [13]:
# SPL: LOGGING f'Learning path has {@path_len} concept(s): {@path}' LEVEL INFO
print('[INFO]', f"Learning path has {path_len} concept(s): {path}")

[INFO] Learning path has 1 concept(s): ['angle']


In [14]:
# SPL: @answer := ''
answer = ''
print(f'@answer =', answer)

@answer = 


In [15]:
# SPL: @spl__i := 0
spl__i = 0
print(f'@spl__i =', spl__i)

@spl__i = 0


In [16]:
# SPL: WHILE @spl__i < @path_len DO
_while_iter = 0
while (spl__i < path_len) and _while_iter < (path_len + 1):
    _while_iter += 1
    # SPL: SOLVE @concept TEXT := @path[@_i]
    concept = path[spl__i]
    print(f'@concept =', concept)
    # SPL: LOGGING f'Section {@_i}: {@concept}' LEVEL INFO
    print('[INFO]', f"Section {spl__i}: {concept}")
    # SPL: CALL cache_get(@concept) INTO @section
    section = cache_get(concept)
    print(f'@section:', section)
    # SPL: EVALUATE @section
    if section == 'miss':
        # SPL: LOGGING f'cache MISS for {@concept} — generating with the LLM' LEVEL INFO
        print('[INFO]', f"cache MISS for {concept} — generating with the LLM")
        # SPL: GENERATE write_section(@concept, @learner_state, @style_guide) INTO @section
        _prompt = WRITE_SECTION_PROMPT.format(concept=concept, context=learner_state, style_guide=style_guide)
        section = _llm_call(_prompt)
        print(f'@section:', section[:200] if isinstance(section, str) else section)
        # SPL: CALL verify_geometry(@section) INTO @check
        check = _verify_geometry(section)
        print(f'@check:', check)
        # SPL: EVALUATE @check
        if "fail" in str(check).lower():
            # SPL: GENERATE refine_section(@section, @check, @style_guide) INTO @section
            _prompt = REFINE_SECTION_PROMPT.format(section=section, feedback=check, style_guide=style_guide)
            section = _llm_call(_prompt)
            print(f'@section:', section[:200] if isinstance(section, str) else section)
        else:
            # SPL: LOGGING 'Geometry verified' LEVEL INFO
            print('[INFO]', 'Geometry verified')
        # SPL: CALL cache_put(@concept, @section) INTO @cache_key
        cache_key = cache_put(concept, section)
        print(f'@cache_key:', cache_key)
        # SPL: LOGGING f'Stored verified section — cache key {@cache_key}' LEVEL INFO
        print('[INFO]', f"Stored verified section — cache key {cache_key}")
    else:
        # SPL: LOGGING f'cache HIT for {@concept} — reusing verified section (0 LLM calls)' LEVEL INFO
        print('[INFO]', f"cache HIT for {concept} — reusing verified section (0 LLM calls)")
    # SPL: @answer := @answer + '\n\n---\n\n' + @section
    answer = answer + '\n\n---\n\n' + section
    print(f'@answer =', answer)
    # SPL: @spl__i := @spl__i + 1
    spl__i = spl__i + 1
    print(f'@spl__i =', spl__i)
# END WHILE  (ran {_while_iter} iteration(s))

@concept = angle
[INFO] Section 0: angle
@section: miss
[INFO] cache MISS for angle — generating with the LLM
@section: \section{Definition of an Angle}

An angle is a fundamental concept in geometry, defined as follows:

Given two rays $OA$ and $OB$, where $\angle AOB$ lies in the interval $(0, \pi)$, we say that the 
@check: pass
[INFO] Geometry verified
@cache_key: spl3:content:b9dbd6b80ad1135f85b3da6b999e494b2c8758108913823929c348324a12bbd1
[INFO] Stored verified section — cache key spl3:content:b9dbd6b80ad1135f85b3da6b999e494b2c8758108913823929c348324a12bbd1
@answer = 

---

\section{Definition of an Angle}

An angle is a fundamental concept in geometry, defined as follows:

Given two rays $OA$ and $OB$, where $\angle AOB$ lies in the interval $(0, \pi)$, we say that the angle measure of $\angle AOB$ is denoted by $\theta$, and we define the measure of an angle to be the ratio of the length of its opposite arc with respect to a circle with center $O$, denoted as $\frac{\text{arc }

In [17]:
# SPL: LOGGING f'Answer complete — {@path_len} section(s), style: {@style}' LEVEL INFO
print('[INFO]', f"Answer complete — {path_len} section(s), style: {style}")

[INFO] Answer complete — 1 section(s), style: textbook


In [18]:
# SPL: COMMIT @answer
_output = answer
_result_path = Path("answer_on_demand_output.md")
_result_path.write_text(_output, encoding='utf-8')
print(f"Committed to {_result_path}")

Committed to answer_on_demand_output.md
